In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="asadshahab/maya-audio", repo_type="dataset", local_dir="./maya-audio")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files: 100%|██████████| 4/4 [00:02<00:00,  1.34it/s]


'/home/ubuntu/maya-audio'

In [3]:
files = glob('maya-audio/data/*.parquet')
files

['maya-audio/data/train-00000-of-00002.parquet',
 'maya-audio/data/train-00001-of-00002.parquet']

In [4]:
df = pd.read_parquet(files[0])
df

,audio,transcript,duration,filename
0,{'bytes': b'RIFF$N\x03\x00WAVEfmt \x10\x00\x00...,People probably wouldn't be arguing about pine...,4.512,output_00001.wav
1,{'bytes': b'RIFF$\x06\x03\x00WAVEfmt \x10\x00\...,"Honestly, though I've heard both sides. It's a...",4.128,output_00002.wav
2,{'bytes': b'RIFF$\xa8\x00\x00WAVEfmt \x10\x00\...,Bigger picture.,0.896,output_00003.wav
3,{'bytes': b'RIFF$\xac\x02\x00WAVEfmt \x10\x00\...,I think humans are naturally inclined to build...,3.648,output_00004.wav
4,{'bytes': b'RIFFD\xd4\x02\x00WAVEfmt \x10\x00\...,It's less about the tech itself. And more abou...,3.862,output_00005+00006.wav
...,...,...,...,...
3743,{'bytes': b'RIFF$\x86\x01\x00WAVEfmt \x10\x00\...,With the crisp scent of leaves.,2.080,output_03764.wav
3744,{'bytes': b'RIFF$p\x05\x00WAVEfmt \x10\x00\x00...,"Hug for your taste buds, that's a beautiful wa...",7.424,output_03765.wav
3745,{'bytes': b'RIFF$\xec\x01\x00WAVEfmt \x10\x00\...,"Rich, decadent, reliably comforting.",2.624,output_03766.wav
3746,{'bytes': b'RIFF$\xd2\x03\x00WAVEfmt \x10\x00\...,Do you prefer yours with frosting? And what ki...,5.216,output_03767.wav


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcript'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"maya_audio"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 3748/3748 [01:22<00:00, 45.42it/s]


In [7]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'maya-audio_audio/maya-audio-data-train-00000-of-00002_0.mp3',
 'text': "People probably wouldn't be arguing about pineapple on pizza with AI companions, for starters.",
 'speaker': 'maya_audio'}

In [8]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'maya-audio')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 338.85ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  278kB /  278kB,  814kB/s  
Processing Files (1 / 1): 100%|██████████|  278kB /  278kB,  694kB/s  
New Data Upload: 100%|██████████|  278kB /  278kB,  694kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.27 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/553148213455b11b6ce7de2821b7c3f63c2cc4cb', commit_message='Upload dataset', commit_description='', oid='553148213455b11b6ce7de2821b7c3f63c2cc4cb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('maya-audio-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [11]:
folders = glob('maya-audio_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

maya-audio_audio
maya-audio_audio_neucodec


In [12]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('maya-audio_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 4.74MB / 4.74MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 4.74MB / 4.74MB,  0.00B/s  
New Data Upload: 100%|██████████| 4.74MB / 4.74MB,  0.00B/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  32%|███▏      | 44.6MB /  141MB,   ???B/s  
Processing Files (0 / 1): 100%|█████████▉|  141MB /  141MB,  481MB/s  
Processing Files (0 / 1): 100%|█████████▉|  141MB /  141MB,  241MB/s  
Processing Files (0 / 1): 100%|█████████▉|  141MB /  141MB,  120MB/s  
Processing Files (1 / 1): 100%|██████████|  141MB /  141MB, 99.6MB/s  
Processing Files (1 / 1): 100%|██████████|  141MB /  141MB, 96.6MB/s  
New Data Upload: 100%|██████████|  141MB /  141MB, 96.6MB/s  
